# Profitable Wallet Analysis

Loads processed trades from `data/trades_processed/` (top-5% wallets by P&L)

In [1]:
import json
import math
from pathlib import Path

import pandas as pd
import numpy as np

## Parameters

In [2]:
TRADES_PROCESSED = Path("../data/trades_processed")

## 1 · Load processed trades

In [ ]:
TRADES_PARQUET = Path("../data/trades_processed.parquet")

if TRADES_PARQUET.exists():
    # Fast path: load from parquet
    df = pd.read_parquet(TRADES_PARQUET)
else:
    # Slow path: load from jsonl files
    rows = []
    for jsonl_path in sorted(TRADES_PROCESSED.rglob("*.jsonl")):
        with jsonl_path.open() as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))

    df = pd.DataFrame(rows)
    df["dt"]               = pd.to_datetime(df["dt"], utc=True)
    df["trade_value_usdc"] = df["trade_value_usdc"].astype(float)
    df["final_value_usdc"] = df["final_value_usdc"].astype(float)
    df["size"]             = df["size"].astype(float)
    df["price"]            = df["price"].astype(float)
    
    # Save for next time
    df.to_parquet(TRADES_PARQUET, index=False)

# calculate pnl
df["pnl"] = np.where(
    df["side"] == "BUY", 
    df["final_value_usdc"] - df["trade_value_usdc"], 
    # we buy the opposite token for (1-p) * size = size - trade_value
    #  final val is (1-final_price) * size = size - final_price * size
    # so pnl is (size - final_price * size) - (size - trade_value) = trade_value - final_price * size
    df["trade_value_usdc"] - df["final_value_usdc"]
)

print(f"Loaded {len(df):,} trade records for {df['wallet'].nunique():,} wallets.")
df.head(3)

Loaded 5,740,685 trade records for 21,244 wallets.


,wallet,condition_id,dt,side,asset,outcome,outcomeIndex,size,price,trade_value_usdc,final_value_usdc,token_winner,final_price,name,pseudonym,transactionHash,title,pnl
0,0xfd5294de1d526489fede36ee0e43ae4779050c2f,0x00f590cd6708167dbf5d13ee797cdbda8871640c207f...,2026-01-01 02:26:01+00:00,BUY,9334307459743296872827532761458378621780912799...,No,1,30.32,0.99,30.0168,30.32,True,1.0,gogogo1234556,Wrong-Fantasy,0x632981ac1c0346d94b71b15878cde1849cae60f8eb31...,Jeremiah Fears: Rebounds Over 3.5,0.3032
1,0x83f842bacf74054747c97fb7688746dcf3d6a32e,0x00fa0f2eb3312f11a143ad960f74fbe899d926bff64b...,2026-01-01 17:33:57+00:00,BUY,5804414565394758982563016809734548947661674807...,Vanderbilt Commodores,0,4.25,0.93,3.9525,4.25,True,1.0,coachstevesports,Kosher-Knee,0xb4602e1f10de01827634cb79cf6d09adc21aa6c8a8f2...,Vanderbilt Commodores vs. Arkansas Razorbacks (W),0.2975
2,0x83f842bacf74054747c97fb7688746dcf3d6a32e,0x00fa0f2eb3312f11a143ad960f74fbe899d926bff64b...,2026-01-01 17:48:07+00:00,BUY,5804414565394758982563016809734548947661674807...,Vanderbilt Commodores,0,5.78,0.96,5.5488,5.78,True,1.0,coachstevesports,Kosher-Knee,0x5d34a862229eb4915cd89e2094e9954359860907be0e...,Vanderbilt Commodores vs. Arkansas Razorbacks (W),0.2312


In [29]:
df[df['side'] == "SELL"].head(3)

,wallet,condition_id,dt,side,asset,outcome,outcomeIndex,size,price,trade_value_usdc,final_value_usdc,token_winner,final_price,name,pseudonym,transactionHash,title,pnl,notional,dt_floored
5,0x9422f0e5a2c4358bdd2a901aa2a31f58e04ef273,0x00fa0f2eb3312f11a143ad960f74fbe899d926bff64b...,2026-01-01 15:00:29+00:00,SELL,6183625064640499871410746675724482888482931718...,Arkansas Razorbacks,1,15.83,0.030,0.4749,0.00,False,0.0,flexer78,Frail-Prayer,0x4420fbe0d790c13a58b6411f3ce5218d66c54fb67ad9...,Vanderbilt Commodores vs. Arkansas Razorbacks (W),0.4749,15.3551,2026-01-01 15:00:00+00:00
7,0x9dd9e18b9dcc393705e73966bc79eccbbd9108bd,0x00fa0f2eb3312f11a143ad960f74fbe899d926bff64b...,2026-01-01 17:58:25+00:00,SELL,5804414565394758982563016809734548947661674807...,Vanderbilt Commodores,0,8.00,0.965,7.7200,8.00,True,1.0,Glued,,0xf8c947bc55e66dc4358655b3b2df5685b6227607c5ab...,Vanderbilt Commodores vs. Arkansas Razorbacks (W),15.7200,0.2800,2026-01-01 17:55:00+00:00
9,0x00270ebfc2d5487e1c6ee922c5dd0ea826ce020b,0x0125bca68397fa9aaa8fc71506f0ad537905a65a3fdd...,2026-01-01 06:57:21+00:00,SELL,1014574915188836130568602917379205857893334077...,Yes,0,36.25,0.994,36.0325,36.25,True,1.0,0x00270eBfc2d5487e1C6ee922C5DD0Ea826CE020B-176...,Loose-Spirituality,0xf43007b80a77c592f1dc207165370da706bb34db2f42...,Will The Fate of Ophelia - Taylor Swift be the...,72.2825,0.2175,2026-01-01 06:55:00+00:00


In [4]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
df.groupby("wallet").agg(
    num_trades = ("transactionHash", "count"),
    num_markets = ("condition_id", "nunique"),
    pnl_usdc    = ("pnl", "sum")
).describe(percentiles=QUANTILES)



,num_trades,num_markets,pnl_usdc
count,21244.000000,21244.000000,2.124400e+04
mean,270.226182,55.344709,2.086577e+04
std,5560.259653,338.437587,1.885485e+05
min,1.000000,1.000000,1.761136e+02
0.1%,1.000000,1.000000,1.769118e+02
1%,1.000000,1.000000,1.837357e+02
5%,1.000000,1.000000,2.145850e+02
10%,1.000000,1.000000,2.725566e+02
25%,2.000000,2.000000,5.513872e+02
50%,7.000000,5.000000,1.462551e+03


## 2 · Weighted-pnl volatility per wallet

Step 1 — collapse fills into **timestamp buckets** `(wallet, dt)`.  
Step 2 — apply the weighted-return volatility formula across those buckets.

In [5]:
import math
import numpy as np
import pandas as pd

def scaled_weighted_pnl_volatility(buckets: pd.DataFrame) -> float:
    """
    Compute capital-weighted PnL volatility scaled by sqrt(total PnL).

    Each row of `buckets` must contain:
        - notional : total capital in the bucket
        - pnl      : realized PnL in the bucket
    """
    if len(buckets) < 2:
        return float("nan")

    w = buckets["notional"].to_numpy()
    pnl = buckets["pnl"].to_numpy()

    total_w = w.sum()
    total_pnl = pnl.sum()

    if total_w == 0 or total_pnl <= 0:
        return float("nan")

    # weighted mean PnL
    mean = np.sum(w * pnl) / total_w

    # weighted variance
    variance = np.sum(w * (pnl - mean) ** 2) / total_w
    sigma = math.sqrt(variance)

    # scale by sqrt of total PnL
    sigma_scaled = sigma / math.sqrt(total_pnl)

    return sigma_scaled

In [6]:
# ── Step 1: aggregate fills → timestamp buckets per wallet ───────────────────
df["notional"] = df.apply(
    lambda r: r["trade_value_usdc"] if r["side"] == "BUY" else r['size'] * (1-r["price"]), 
    axis=1
)

# floor to the nearest 5 minutes
df["dt_floored"] = df["dt"].dt.floor("5T") 

buckets = (
    df.groupby(["wallet", "dt_floored", "condition_id"], sort=False)
    .agg(
        notional=("notional", "sum"),
        pnl=("pnl", "sum"),
    )
    .reset_index()
)

# drop buckets with zero notional (e.g. dust / zero-cost trades)
buckets = buckets[buckets["notional"] > 0].copy()

print(f"Timestamp buckets: {len(buckets):,}  across {buckets['wallet'].nunique():,} wallets.")
buckets.head(5)

/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_11967/1310151840.py:8: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df["dt_floored"] = df["dt"].dt.floor("5T")


Timestamp buckets: 3,538,930  across 21,244 wallets.


,wallet,dt_floored,condition_id,notional,pnl
0,0xfd5294de1d526489fede36ee0e43ae4779050c2f,2026-01-01 02:25:00+00:00,0x00f590cd6708167dbf5d13ee797cdbda8871640c207f...,30.0168,0.3032
1,0x83f842bacf74054747c97fb7688746dcf3d6a32e,2026-01-01 17:30:00+00:00,0x00fa0f2eb3312f11a143ad960f74fbe899d926bff64b...,3.9525,0.2975
2,0x83f842bacf74054747c97fb7688746dcf3d6a32e,2026-01-01 17:45:00+00:00,0x00fa0f2eb3312f11a143ad960f74fbe899d926bff64b...,5.5488,0.2312
3,0x83f842bacf74054747c97fb7688746dcf3d6a32e,2026-01-01 18:00:00+00:00,0x00fa0f2eb3312f11a143ad960f74fbe899d926bff64b...,121.2500,3.7500
4,0x83f842bacf74054747c97fb7688746dcf3d6a32e,2026-01-01 18:15:00+00:00,0x00fa0f2eb3312f11a143ad960f74fbe899d926bff64b...,122.5000,2.5000


In [7]:
WALLET = "0xa49becb692927d455924583b5e3e5788246f4c40"

print("PnL: " + str(df[df['wallet'] == WALLET]['pnl'].sum()))

(buckets[buckets["wallet"] == WALLET]
 .groupby("wallet")
 .apply(scaled_weighted_pnl_volatility, include_groups=False)
 .head(5))

PnL: 0.0


,dt_floored,condition_id,notional,pnl
wallet,,,,


In [8]:
# ── Step 2: volatility per wallet ────────────────────────────────────────────

def wallet_metrics(group: pd.DataFrame) -> pd.Series:
    """Compute all wallet metrics in one pass."""
    pnl = group["pnl"].to_numpy()
    total_notional = group["notional"].sum()
    total_pnl = pnl.sum()
    
    # top-5 pnl concentration
    if total_pnl <= 0:
        top5_pnl_pct = float("nan")
        top_market_pnl_pct = float("nan")
    else:
        top5_pnl = np.sort(pnl)[-5:].sum()
        top5_pnl_pct = top5_pnl / total_pnl
        top_market_pnl_pct = group.groupby("condition_id")["pnl"].sum().max() / total_pnl
    
    return pd.Series({
        "pnl_volatility": scaled_weighted_pnl_volatility(group),
        "num_buckets": len(group),
        "num_markets": len(group['condition_id'].unique()),
        "total_notional": total_notional,
        "total_pnl": total_pnl,
        "top5_pnl_pct": top5_pnl_pct,
        "top_market_pnl_pct": top_market_pnl_pct,
    })

wallet_vol = (
    buckets.groupby("wallet", sort=False)
    .apply(wallet_metrics, include_groups=False)
    .reset_index()
)

wallet_vol["return"] = wallet_vol["total_pnl"] / wallet_vol["total_notional"]
wallet_vol = wallet_vol.sort_values("pnl_volatility").reset_index(drop=True)

print(f"Wallets with volatility: {wallet_vol['pnl_volatility'].notna().sum():,}")
wallet_vol.head(10)

Wallets with volatility: 17,250


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return
0,0x3a42935c95bee13777f7c322518ab47cb06110ab,0.000000,2.0,1.0,99.999998,354.545454,1.000000,1.000000,3.545455
1,0x9be69b8ebae6f8279dccaf7123e6c98769a37535,0.001776,2.0,2.0,8477.765440,710.064560,1.000000,0.507334,0.083756
2,0x9803b7b4d2c33103cc0e3a57f1fb664a39447850,0.005675,2.0,1.0,2021.109996,718.110946,1.000000,1.000000,0.355305
3,0x3e588ff146eb4daea133af7bc2c14399e84d80c1,0.014752,2.0,2.0,1320.810000,1124.190000,1.000000,0.500440,0.851137
4,0x31a8f18deafa62fbf9cc89f6f5d95aab53970936,0.019497,1498.0,1494.0,79666.162790,223.127210,0.391797,0.177477,0.002801
5,0xee4d91b7d0fad963c2a4bf7b53b6bfb0990ecae5,0.022542,2.0,2.0,441.741800,313.278200,1.000000,0.501274,0.709188
6,0x15bd8329499e9df59d863dfddb823228ebcc1b0e,0.024192,2.0,2.0,161.120400,198.119600,1.000000,0.501788,1.229637
7,0xf91282f2838fbd67836c554709403e291d70d178,0.024813,2.0,2.0,2497.113000,4528.887000,1.000000,0.508734,1.813649
8,0xf0ced243f3396ddca467b74a2fde91f153107713,0.029043,3.0,3.0,2261.210000,5622.790000,1.000000,0.342293,2.486629
9,0x61002f423f644853d492e65c8b77e91098a42120,0.029922,2.0,2.0,1182.747200,1222.612800,1.000000,0.500862,1.033706


In [9]:
wallet_vol[wallet_vol["wallet"] == WALLET]

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return


In [10]:
# df[df["wallet"] == WALLET][['dt', 'title', 'side', 'size', 'price', 'pnl']].sort_values('dt')

## 3 · Volatility distribution

In [11]:
QUANTILES = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
N = wallet_vol["pnl_volatility"].notna().sum()

vol_stats = (
    wallet_vol["pnl_volatility"]
    .dropna()
    .quantile(QUANTILES)
    .rename_axis("quantile")
    .to_frame()
)
vol_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
vol_stats.loc["mean"] = [float("nan"), wallet_vol["pnl_volatility"].mean()]
vol_stats.loc["std"]  = [float("nan"), wallet_vol["pnl_volatility"].std()]
vol_stats["wallet_count"] = vol_stats["wallet_count"].astype("Int64")
vol_stats["pnl_volatility"] = vol_stats["pnl_volatility"].round(4)

vol_stats

,wallet_count,pnl_volatility
quantile,,
0.01,172,0.2423
0.05,862,0.6522
0.1,1725,0.9907
0.25,4312,2.0234
0.5,8625,4.3852
0.75,12938,10.1652
0.9,15525,21.9509
0.95,16388,31.7074
0.99,17078,70.3593


In [12]:
len(wallet_vol)

21244

In [13]:
len(wallet_vol[(wallet_vol['num_buckets'] >= 20) &
               (wallet_vol['top5_pnl_pct'] <= 0.6) 
               & (wallet_vol['return'] >= 0.1)
               ])

3020

In [14]:
filtered_wallets = wallet_vol[
    (wallet_vol['num_buckets'] >= 20) &
    (wallet_vol['num_buckets'] <= 300) & 
    (wallet_vol['top5_pnl_pct'] <= 0.4) &
    (wallet_vol['top_market_pnl_pct'] <= 0.5) &
    (wallet_vol['return'] >= 0.1)
].sort_values("pnl_volatility")

print(f"Filtered wallets: {len(filtered_wallets):}")

filtered_wallets.head(20)

Filtered wallets: 882


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return
95,0xb0d135ae5677a9a1e638f9da09df7f24c4f639f8,0.151363,49.0,33.0,2157.273404,280.626362,0.368426,0.129496,0.130084
123,0xdeb264d4ee0fcb83d86d0eb242cca86d4a5b9aa8,0.191172,138.0,98.0,126029.011805,54156.645654,0.309983,0.092279,0.429716
129,0x9335b78167d096648f9b5df74d46efd45ce00c21,0.195494,131.0,81.0,2462.420310,3780.909690,0.239093,0.069456,1.535444
130,0x4a0cd23b4d0e2db1c5664146c42fed57f6243843,0.195774,294.0,167.0,4505.082213,8330.717955,0.119362,0.031363,1.849182
156,0x0d7c5eb29ca973c0bf0184bd035a0671d88d3b0c,0.225280,52.0,31.0,11894.385749,19295.473009,0.275244,0.057624,1.622234
188,0x404c04ab8622f35f6dadef2af854af1cc97c74dc,0.260328,182.0,108.0,9392.202580,10136.627420,0.326805,0.101559,1.079260
198,0xd391d4c98709f61f72472cbd8e8126fc1e3c093f,0.269884,201.0,117.0,9163.210876,7414.389926,0.330921,0.097540,0.809148
199,0x69f2adf6e1cac2b54e9f511adb8dc44436e38045,0.270464,110.0,38.0,13896.425394,25009.023911,0.315670,0.087176,1.799673
217,0x889d3b5fd1a283980fb09eec122585d4339ee36b,0.284958,282.0,180.0,4239.011081,6689.010924,0.154340,0.050981,1.577965
218,0x3cc98016d457900d47f78320876d18eb216c185b,0.284971,173.0,137.0,1178.287933,2294.487301,0.165074,0.046450,1.947306


In [15]:
# print deciles of pnl_volatility for filtered wallets

for i in range(0, 11):
    q = i / 10
    vol_q = filtered_wallets["pnl_volatility"].quantile(q)
    print(f"  Volatility at {q:.0%} pct: {vol_q:.4f}")

  Volatility at 0% pct: 0.1514
  Volatility at 10% pct: 0.5294
  Volatility at 20% pct: 0.7174
  Volatility at 30% pct: 0.9520
  Volatility at 40% pct: 1.1986
  Volatility at 50% pct: 1.4647
  Volatility at 60% pct: 1.8061
  Volatility at 70% pct: 2.2830
  Volatility at 80% pct: 2.9201
  Volatility at 90% pct: 4.4273
  Volatility at 100% pct: 21.8362


In [16]:
wallet_vol[wallet_vol['num_buckets'] >= 2].sort_values("num_buckets").head(10)

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return
0,0x3a42935c95bee13777f7c322518ab47cb06110ab,0.000000,2.0,1.0,99.999998,354.545454,1.0,1.000000,3.545455
12095,0x6afc9eb95e2df059728e7b6f666b31361874db4c,8.428189,2.0,2.0,975.070000,431.930000,1.0,1.205265,0.442973
7592,0xed64a5e4553b233135f43706a7b3c95ed81262e3,3.652725,2.0,2.0,508.320000,508.320000,1.0,0.672627,1.000000
12096,0xc05c1b9df9d33d88684a00e5735401dd732ec739,8.429199,2.0,2.0,2118.000000,2282.000000,1.0,0.968449,1.077432
12099,0x8b32d57e2a6c257c2431a7b416d598dc2ff6618e,8.435421,2.0,2.0,2210.000000,2040.000000,1.0,0.704706,0.923077
2688,0x99435844497964309fdbf431a68aa1ff6eb463e6,1.359652,2.0,2.0,145.217200,181.532800,1.0,1.007972,1.250078
7579,0x43a481dd0e194f8582d22185f7508d9ac0b3aac7,3.645766,2.0,2.0,439.700000,518.020000,1.0,0.735416,1.178121
14961,0xadb5d7d430d292df7dc166b7523b6c96551b87de,18.110654,2.0,2.0,3123.410000,1857.590000,1.0,1.179830,0.594731
2694,0x736fb8ecf37278c1c666be2438c63472164cb5fb,1.362006,2.0,2.0,521.999998,478.000002,1.0,1.004184,0.915709
12105,0x8232600347ea7847d02f4180207f2bc20bcf89e0,8.447610,2.0,2.0,3165.997240,540.922760,1.0,0.985353,0.170854


In [17]:
# # display all rows
# pd.set_option('display.max_rows', None)

# buckets[
#     (buckets['wallet'] == WALLET) 
#     # (buckets['dt'] >= pd.Timestamp('2026-01-03T19:00:00', tz='UTC')) &
#     # (buckets['dt'] < pd.Timestamp('2026-01-04', tz='UTC'))
#      ].sort_values("dt_floored")

In [18]:
import plotly.express as px

# Select top 10 wallets by total_pnl with at least 100 buckets
# top_wallets = (
#     wallet_vol[wallet_vol['num_buckets'] >= 100]
#     .nlargest(10, 'total_pnl')['wallet']
#     .tolist()
# )
top_wallets = filtered_wallets.head(20)['wallet'].tolist()

# Filter buckets for these wallets and compute cumulative PnL over time
cumulative_pnl = (
    buckets[buckets['wallet'].isin(top_wallets)]
    .sort_values(['wallet', 'dt_floored'])
    .copy()
)
cumulative_pnl['cumulative_pnl'] = cumulative_pnl.groupby('wallet')['pnl'].cumsum()

# Plot
fig = px.line(
    cumulative_pnl,
    x='dt_floored',
    y='cumulative_pnl',
    color='wallet',
    title='Cumulative PnL Over Time by Wallet',
    labels={'dt_floored': 'Time', 'cumulative_pnl': 'Cumulative PnL (USDC)', 'wallet': 'Wallet'}
)
fig.show(renderer='browser')

In [19]:
top_wallet_set = set(filtered_wallets['wallet'])

top_wallet_df=df[df['wallet'].isin(top_wallet_set)] 

print(f"len(top_wallet_df): {len(top_wallet_df):}")

top_wallet_df[['dt', 'title', 'pseudonym', 'condition_id', 'wallet', 'size', 'price', 'pnl']].sort_values('dt').head(10)

len(top_wallet_df): 167178


,dt,title,pseudonym,condition_id,wallet,size,price,pnl
2847940,2025-10-09 01:38:34+00:00,Will Marques Brownlee's next video get between...,Profuse-Restriction,0x4dd76ac10cb49bdbe7c7df792534f84dc244bbb9b638...,0x796301e7bb7bb0891993b4b35ca2bdc10fe35c7b,28.000000,0.910000,2.520000
2847941,2025-10-09 01:49:26+00:00,Will Marques Brownlee's next video get between...,Profuse-Restriction,0x4dd76ac10cb49bdbe7c7df792534f84dc244bbb9b638...,0x796301e7bb7bb0891993b4b35ca2bdc10fe35c7b,20.000000,0.920000,1.600000
2836128,2025-10-09 02:28:28+00:00,Will Marques Brownlee's next video get 3.5 mil...,Profuse-Restriction,0x44192dcadfce1b11f2b986149dff1e2e92d565aeea6f...,0x796301e7bb7bb0891993b4b35ca2bdc10fe35c7b,10.000000,0.951000,0.490000
2980914,2025-10-09 03:05:52+00:00,Will Marques Brownlee's next video get less th...,Profuse-Restriction,0xa5f117032e376a8a7330d83f7b2a8abf7c53a0b92945...,0x796301e7bb7bb0891993b4b35ca2bdc10fe35c7b,9.205881,0.340000,6.075882
2980915,2025-10-09 03:13:34+00:00,Will Marques Brownlee's next video get less th...,Profuse-Restriction,0xa5f117032e376a8a7330d83f7b2a8abf7c53a0b92945...,0x796301e7bb7bb0891993b4b35ca2bdc10fe35c7b,40.000000,0.290000,28.400000
2980916,2025-10-09 04:49:03+00:00,Will Marques Brownlee's next video get less th...,Profuse-Restriction,0xa5f117032e376a8a7330d83f7b2a8abf7c53a0b92945...,0x796301e7bb7bb0891993b4b35ca2bdc10fe35c7b,59.200000,0.147973,67.960000
3076571,2025-10-09 05:42:15+00:00,Will Marques Brownlee's next video get between...,Profuse-Restriction,0xdd2f6a958de5d007541509b86d6ab222254a280f7af6...,0x796301e7bb7bb0891993b4b35ca2bdc10fe35c7b,79.411763,0.340000,52.411764
2980917,2025-10-09 05:45:51+00:00,Will Marques Brownlee's next video get less th...,Profuse-Restriction,0xa5f117032e376a8a7330d83f7b2a8abf7c53a0b92945...,0x796301e7bb7bb0891993b4b35ca2bdc10fe35c7b,36.820511,0.380223,22.820512
2980918,2025-10-09 05:47:59+00:00,Will Marques Brownlee's next video get less th...,Profuse-Restriction,0xa5f117032e376a8a7330d83f7b2a8abf7c53a0b92945...,0x796301e7bb7bb0891993b4b35ca2bdc10fe35c7b,13.150000,0.380228,8.150000
2836129,2025-10-09 05:51:53+00:00,Will Marques Brownlee's next video get 3.5 mil...,Profuse-Restriction,0x44192dcadfce1b11f2b986149dff1e2e92d565aeea6f...,0x796301e7bb7bb0891993b4b35ca2bdc10fe35c7b,10.000000,0.997000,19.970000


In [24]:
top_wallet_df.groupby("condition_id").agg(
    title = ("title", "first"),
    total_pnl = ("pnl", "sum"),
    num_trades = ("transactionHash", "count"),
    total_notional = ("notional", "sum"),
    wallet_count = ("wallet", "count")
).sort_values("total_pnl", ascending=False).head(30)

,title,total_pnl,num_trades,total_notional,wallet_count
condition_id,,,,,
0x74d786eb72f4de44cb39da160a401bcd1a7c390ae03963c1dc13565cc337b9f5,Titans vs. Jaguars,352382.110928,23,143433.615467,23
0x8cbe26c43817d74689889e24f5faf01ff12a0e5d358997510622b2d1730f5640,Michigan Wolverines vs. Penn State Nittany Lions,348290.650541,13,23982.050868,13
0x4df5dcf303454c2087067fc4b27201a741707fb8dd768e6d96c24c7eb7adea63,Suns vs. Grizzlies,323759.909143,46,119800.214522,46
0xb8c1bd306a8a4cedfb280e114e655c5092b3f37edccae05cd877d7f21a5774ce,"Russia x Ukraine ceasefire by January 31, 2026?",220250.062475,13,3814.901196,13
0xea49f0969abc80cf7ddbc9d3435c81d3a63cc6b73ebf8d6420f59b0557cf188e,Will Real Madrid CF win on 2026-01-28?,189604.557576,54,17526.151310,54
0xf4b9279a3656fe7006118dc51037ebe28bdd3090e8a4127889546315f6b6e2f7,Will Manchester City FC vs. Brighton & Hove Al...,187452.907105,26,14800.840933,26
0x69a9f82946db864ef134bece1e61b82c2dedfaeef4b8f1bffd1d1cecba621cc6,Counter-Strike: Spirit vs FURIA - Map 1 Winner,173197.286118,114,51290.760376,114
0xe1c1cfe25fa7d982a943264579081767708594952ea53ba3db8888d6fa79d853,Pelicans vs. Bulls,171730.896739,154,97967.068618,154
0xea80d698b87967427287ce261013115888fba0a814a47b6fc622baa744798079,"UFC 324: Pulyaev vs. Gautier (Middleweight, Pr...",165738.401174,31,57823.400790,31


In [28]:
top_wallet_df[top_wallet_df['condition_id'] == "0x74d786eb72f4de44cb39da160a401bcd1a7c390ae03963c1dc13565cc337b9f5"]

,wallet,condition_id,dt,side,asset,outcome,outcomeIndex,size,price,trade_value_usdc,final_value_usdc,token_winner,final_price,name,pseudonym,transactionHash,title,pnl,notional,dt_floored
273027,0x0362bb926368b144e0ff98f6828a251e4cb6449e,0x74d786eb72f4de44cb39da160a401bcd1a7c390ae039...,2025-12-30 18:06:35+00:00,SELL,7130346841273377112398274928811291939225008666...,Jaguars,1,6.470000,0.850000,5.499500,6.470000,True,1.0,varch01,Straight-Apprehension,0x33b33908340e3b47085c1bacd00906fa560313968002...,Titans vs. Jaguars,11.969500,0.970500,2025-12-30 18:05:00+00:00
273035,0x13ce0f6f73a83a2be1eb484c91169c2bd5b935f0,0x74d786eb72f4de44cb39da160a401bcd1a7c390ae039...,2026-01-04 18:12:33+00:00,BUY,7130346841273377112398274928811291939225008666...,Jaguars,1,159.420000,0.740000,117.970800,159.420000,True,1.0,mozak,Nifty-Sunlamp,0x6d058b11ad208550c05c165d33aa37abbb20c4a30f93...,Titans vs. Jaguars,41.449200,117.970800,2026-01-04 18:10:00+00:00
273036,0x13ce0f6f73a83a2be1eb484c91169c2bd5b935f0,0x74d786eb72f4de44cb39da160a401bcd1a7c390ae039...,2026-01-04 20:47:35+00:00,SELL,7130346841273377112398274928811291939225008666...,Jaguars,1,159.410000,0.999000,159.250590,159.410000,True,1.0,mozak,Nifty-Sunlamp,0xe607674322003a98a4eee27e79b139b13fa4ed0adb54...,Titans vs. Jaguars,318.660590,0.159410,2026-01-04 20:45:00+00:00
273044,0x1a7ac90faa333c269a1e1640c9e2efd04c417181,0x74d786eb72f4de44cb39da160a401bcd1a7c390ae039...,2026-01-04 18:47:23+00:00,BUY,4730742259843752557766821530226910872007181236...,Titans,0,278.983750,0.071689,20.000000,0.000000,False,0.0,Nas84950,Offbeat-Airfield,0xd1b4fecb2272b69ffc34c2ae0204ab19bd27b77aa33c...,Titans vs. Jaguars,-20.000000,20.000000,2026-01-04 18:45:00+00:00
273090,0x49e20e64433eac734dd6024352b007281012018d,0x74d786eb72f4de44cb39da160a401bcd1a7c390ae039...,2026-01-04 16:30:29+00:00,BUY,4730742259843752557766821530226910872007181236...,Titans,0,500.000000,0.120000,60.000000,0.000000,False,0.0,d39f,Which-Strand,0x2167192377a45208be53b34b17b2473f48e729c4b646...,Titans vs. Jaguars,-60.000000,60.000000,2026-01-04 16:30:00+00:00
273097,0x4f5b608cbdaac7d946578a58493109ed9a60d411,0x74d786eb72f4de44cb39da160a401bcd1a7c390ae039...,2026-01-04 21:04:15+00:00,SELL,7130346841273377112398274928811291939225008666...,Jaguars,1,6600.000000,0.999000,6593.400000,6600.000000,True,1.0,hohoit,That-Debt,0xda1451d576c16996bf2047bf3a0c28e7d3aecd969b64...,Titans vs. Jaguars,13193.400000,6.600000,2026-01-04 21:00:00+00:00
273106,0x65d4d905958b418a30c9f72046208689ae094e71,0x74d786eb72f4de44cb39da160a401bcd1a7c390ae039...,2026-01-04 18:18:43+00:00,BUY,7130346841273377112398274928811291939225008666...,Jaguars,1,773.655593,0.833304,644.689998,773.655593,True,1.0,bo0xts,International-Winner,0xd7b82e82dd79e46cf47f46209e1c9a6b0907488e1376...,Titans vs. Jaguars,128.965595,644.689998,2026-01-04 18:15:00+00:00
273107,0x65d4d905958b418a30c9f72046208689ae094e71,0x74d786eb72f4de44cb39da160a401bcd1a7c390ae039...,2026-01-04 19:47:15+00:00,SELL,7130346841273377112398274928811291939225008666...,Jaguars,1,773.650000,0.998000,772.102700,773.650000,True,1.0,bo0xts,International-Winner,0xcdb6f84e30b100defb450010c0dd3c8962b3cd451e3b...,Titans vs. Jaguars,1545.752700,1.547300,2026-01-04 19:45:00+00:00
273119,0x769056fb242c80b64ac3b74a8161a70468fd74e1,0x74d786eb72f4de44cb39da160a401bcd1a7c390ae039...,2026-01-04 16:29:27+00:00,BUY,7130346841273377112398274928811291939225008666...,Jaguars,1,38995.680000,0.870000,33926.241600,38995.680000,True,1.0,mudman88,Cold-Trousers,0xf5f4fa191717f32be672505c9868964d541d70ccc90f...,Titans vs. Jaguars,5069.438400,33926.241600,2026-01-04 16:25:00+00:00
273120,0x769056fb242c80b64ac3b74a8161a70468fd74e1,0x74d786eb72f4de44cb39da160a401bcd1a7c390ae039...,2026-01-04 16:29:51+00:00,BUY,7130346841273377112398274928811291939225008666...,Jaguars,1,87.961925,0.875379,76.999994,87.961925,True,1.0,mudman88,Cold-Trousers,0xead34e7bc08b2eeb34682af20f5c5b4131b72000be9c...,Titans vs. Jaguars,10.961931,76.999994,2026-01-04 16:25:00+00:00
